# **Inferring**
In this lesson, you will infer sentiment and topics from product reviews and news articles.

## Setup

In [1]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [2]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

def get_completion(prompt, model="gpt-3.5-turbo"):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0, # this is the degree of randomness of the model's output
    )
    return response.choices[0].message.content

## Product review text

In [3]:
lamp_review = """
Needed a nice lamp for my bedroom, and this one had \
additional storage and not too high of a price point. \
Got it fast.  The string to our lamp broke during the \
transit and the company happily sent over a new one. \
Came within a few days as well. It was easy to put \
together.  I had a missing part, so I contacted their \
support and they very quickly got me the missing piece! \
Lumina seems to me to be a great company that cares \
about their customers and products!!
"""

## Sentiment (positive/negative)

In [4]:
prompt = f"""
What is the sentiment of the following product review, 
which is delimited with triple backticks?

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

The sentiment of the review is positive. The reviewer is satisfied with the lamp, the customer service, and the company in general.


In [5]:
prompt = f"""
What is the sentiment of the following product review, 
which is delimited with triple backticks?

Give your answer as a single word, either "positive" \
or "negative".

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

Positive


## Identify types of emotions

In [6]:
prompt = f"""
Identify a list of emotions that the writer of the \
following review is expressing. Include no more than \
five items in the list. Format your answer as a list of \
lower-case words separated by commas.

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

happy, satisfied, grateful, impressed, content


## Identify anger

In [7]:
prompt = f"""
Is the writer of the following review expressing anger?\
The review is delimited with triple backticks. \
Give your answer as either yes or no.

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

No


## Extract product and company name from customer reviews

In [8]:
prompt = f"""
Identify the following items from the review text: 
- Item purchased by reviewer
- Company that made the item

The review is delimited with triple backticks. \
Format your response as a JSON object with \
"Item" and "Brand" as the keys. 
If the information isn't present, use "unknown" \
as the value.
Make your response as short as possible.
  
Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

{
  "Item": "lamp",
  "Brand": "Lumina"
}


## Doing multiple tasks at once

In [9]:
prompt = f"""
Identify the following items from the review text: 
- Sentiment (positive or negative)
- Is the reviewer expressing anger? (true or false)
- Item purchased by reviewer
- Company that made the item

The review is delimited with triple backticks. \
Format your response as a JSON object with \
"Sentiment", "Anger", "Item" and "Brand" as the keys.
If the information isn't present, use "unknown" \
as the value.
Make your response as short as possible.
Format the Anger value as a boolean.

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

{
    "Sentiment": "positive",
    "Anger": false,
    "Item": "lamp",
    "Brand": "Lumina"
}


## Inferring Text Topics
Another application inferring by an LLM is deducing topics from a lengthy piece of text.

This time, the sample is regarding a fictitious newspaper article about a survey conducted by the government measuring the satisfaction rate of workers in government agencies. The results reveal that NASA workers had the highest satisfaction rating.Inferring Text Topics
Another application inferring by an LLM is deducing topics from a lengthy piece of text.

This time, the sample is regarding a fictitious newspaper article about a survey conducted by the government measuring the satisfaction rate of workers in government agencies. The results reveal that NASA workers had the highest satisfaction rating.

In [10]:
story = """
In a recent survey conducted by the government, 
public sector employees were asked to rate their level 
of satisfaction with the department they work at. 
The results revealed that NASA was the most popular 
department with a satisfaction rating of 95%.

One NASA employee, John Smith, commented on the findings, 
stating, "I'm not surprised that NASA came out on top. 
It's a great place to work with amazing people and 
incredible opportunities. I'm proud to be a part of 
such an innovative organization."

The results were also welcomed by NASA's management team, 
with Director Tom Johnson stating, "We are thrilled to 
hear that our employees are satisfied with their work at NASA. 
We have a talented and dedicated team who work tirelessly 
to achieve our goals, and it's fantastic to see that their 
hard work is paying off."

The survey also revealed that the 
Social Security Administration had the lowest satisfaction 
rating, with only 45% of employees indicating they were 
satisfied with their job. The government has pledged to 
address the concerns raised by employees in the survey and 
work towards improving job satisfaction across all departments.
"""

Five topics discussed in the article are requested from the model in a format that each item is one or two words long and in a comma-separated list. ChatGPT returns the topics as government surveys, job satisfaction, NASA, etc.

In [11]:
prompt = f"""
Determine five topics that are being discussed in the \
following text, which is delimited by triple backticks.

Make each item one or two words long. 

Format your response as a list of items separated by commas without numbering them.

Text sample: '''{story}'''
"""
response = get_completion(prompt)
print(response)

Government survey, Employee satisfaction, NASA, Social Security Administration, Job satisfaction


In [12]:
response.split(sep=', ')

['Government survey',
 'Employee satisfaction',
 'NASA',
 'Social Security Administration',
 'Job satisfaction']

## Make a news alert for certain topics

The final sample application is about the selection of topics that a text covers, among a targeted topics list. Initially, the list of possible topics is defined:The final sample application is about the selection of topics that a text covers, among a targeted topics list. Initially, the list of possible topics is defined:

In [13]:
topic_list = [
    "nasa", "local government", "engineering", 
    "employee satisfaction", "federal government"
]

In [14]:
prompt = f"""
Determine whether each item in the following list of \
topics is a topic in the text below, which
is delimited with triple backticks.

Give your answer as a dictionay where the key is a topic and the value is 0 or 1 for each topic if it appears.\

List of topics: {", ".join(topic_list)}

Text sample: '''{story}'''
"""
response = get_completion(prompt)
print(response)

{
    "nasa": 1,
    "local government": 0,
    "engineering": 0,
    "employee satisfaction": 1,
    "federal government": 1
}


In [15]:
for i in response.split(', '):
    print(i)

{
    "nasa": 1,
    "local government": 0,
    "engineering": 0,
    "employee satisfaction": 1,
    "federal government": 1
}


In [16]:
topic_dict = {i.split(': ')[0]: int(i.split(': ')[1]) for i in response.split(sep=', ')}
if topic_dict['nasa'] == 1:
    print("ALERT: New NASA story!")

ValueError: invalid literal for int() with base 10: '1,\n    "local government"'

# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [17]:
# Version 1: Customer Routing & Mood Inference
prompt_1 = f"""
Analyze the following product review delimited by triple backticks.
Infer the following information:
1. Customer Mood: (e.g., thrilled, annoyed, neutral)
2. Priority Level: (Low, Medium, High - based on whether they need immediate help)
3. Recommended Department: (e.g., Sales, Support, Ignore)

Format the output strictly as a JSON object with the keys "Mood", "Priority", and "Department".

Review text: '''{lamp_review}'''
"""
response_1 = get_completion(prompt_1)
print("--- Version 1: Customer Routing ---")
print(response_1)

--- Version 1: Customer Routing ---
{
  "Mood": "Thrilled",
  "Priority": "Medium",
  "Department": "Support"
}


In [18]:
# Version 2: Bias and Fact Checking
prompt_2 = f"""
Analyze the news article delimited by triple backticks.
Infer the following:
- Tone of the article: (1-2 words)
- Does the article mention budget cuts? (True or False)
- Which department had the lowest satisfaction? 

Format your response as a simple list.

Article text: '''{story}'''
"""
response_2 = get_completion(prompt_2)
print("\n--- Version 2: News Article Analysis ---")
print(response_2)


--- Version 2: News Article Analysis ---
- Tone of the article: Positive
- Does the article mention budget cuts? False
- Which department had the lowest satisfaction? Social Security Administration


In [19]:
# Version 3: Sarcasm and Customer Retention
prompt_3 = f"""
Read the review delimited by triple backticks and answer these two questions:
1. Is the reviewer being sarcastic or passive-aggressive? (Yes or No, with a 1-sentence reason)
2. Retention Score: On a scale of 1-10, how likely is this customer to buy from Lumina again?

Review text: '''{lamp_review}'''
"""
response_3 = get_completion(prompt_3)
print("\n--- Version 3: Sarcasm & Retention ---")
print(response_3)


--- Version 3: Sarcasm & Retention ---
1. No, the reviewer is not being sarcastic or passive-aggressive. The reviewer is genuinely satisfied with the product and the company's customer service.
2. Retention Score: 10. This customer is highly likely to buy from Lumina again based on their positive experience with the product and customer service.


What I tested
In this lab, I used the OpenAI API to see how well GPT-3.5 can "read between the lines" and infer information that isn't explicitly stated in the text. I tested this by having the AI act as a customer routing system, a fact-checker for news articles, and a sarcasm detector.

My findings
Overall, the model is incredibly good at inferring human emotions and context, which is something traditional coding can't really do.

In version 1, I gave it a product review where a part broke, but customer service fixed it fast. The model correctly inferred that the customer's mood was "Thrilled" and formatted the whole response as a clean JSON object that could easily be plugged into a database.

In version 3, I asked it to check for sarcasm. It correctly identified that the reviewer was being totally genuine, and it gave them a 10/10 retention score because they loved the customer service.

Testing for hallucinations
In previous labs, the model would sometimes invent data just to fill out a pattern. In version 2, I deliberately tried to trick the model by asking if a specific news article mentioned "budget cuts" (it didn't). This time, the model didn't hallucinate at all. It correctly responded "False" and pulled the exact right data about the Social Security Administration having the lowest satisfaction rating. It seems like inferring facts from a provided text is much safer than asking it to generate new text.

What I learned

AI gets human tone: Trying to write standard Python code to detect sarcasm or infer a "thrilled" mood would be almost impossible. LLMs handle this easily just by reading the context.

JSON formatting is reliable: You can easily force the model to spit out its inferences as JSON keys and values, which makes it super easy to connect these AI prompts directly to a real app (like routing tickets to different departments).

Prompting for facts reduces hallucinations: When you give the model a block of text and strictly ask it "true or false" questions about that text, it is way less likely to make things up compared to when you ask it to summarize or write a story.